# Week 5 Breakout Slides: Interactive Modeling

The Linear Regression and Decision Tree notebooks each walked through one fixed workflow. Here we make the workflow *interactive*: a small dashboard, built with `ipywidgets` and `plotly`, lets you pick the **target**, choose **which features** feed the model, and switch between **model types** — then see the fit update instantly. In the second half we open up **cross-validation**, visualizing how different splitting strategies carve the data into train and validation sets.

> **Requirements.** This notebook needs `plotly` and `ipywidgets` and must run in a live Jupyter kernel — the widgets and Plotly charts won't render in a static preview. If they aren't installed yet: `pip install plotly ipywidgets`.

## Import Libraries

Alongside `numpy`/`pandas`, we bring in `plotly` for interactive charts, `ipywidgets` for the controls, and the `scikit-learn` models, splitters, and metrics we'll wire into the dashboard.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, ShuffleSplit, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Load the Data

The same country development dataset from Week 4 — one row per country, all numeric columns, so any column can serve as the target or a feature.

In [ ]:
path_to_file = Path('your/path')

In [ ]:
df = pd.read_csv(path_to_file / 'country_data.csv')

### Inspect the DataFrame

In [ ]:
df.head()

## An Interactive Model Explorer

We'll assemble three controls, hand them to a single fit-and-plot function, and let `ipywidgets` re-run that function automatically whenever a control changes.

### Build the Widgets

- **Target** — a dropdown of every column; the one we try to predict.
- **Features** — a multi-select (hold Ctrl/Cmd to pick several) of the inputs. Whatever you choose as the target is automatically excluded from the features, so it can never leak into the model.
- **Model** — switch between linear regression and a depth-3 decision tree.

In [ ]:
target_dropdown = widgets.Dropdown(
    options=list(df.columns),
    value='life_expec',
    description='Target:',
)

feature_select = widgets.SelectMultiple(
    options=list(df.columns),
    value=tuple(c for c in df.columns if c != 'life_expec'),
    description='Features:',
    rows=len(df.columns),
)

model_dropdown = widgets.Dropdown(
    options=['Linear Regression', 'Decision Tree'],
    value='Linear Regression',
    description='Model:',
)

### Define the Fit-and-Plot Function

The function reads the current control values, trains on an 80/20 split, and draws an interactive predicted-vs-actual chart. Points on the dashed line are perfect predictions; the title reports R², MAE, and RMSE so you can watch the scores move as you change the inputs.

In [ ]:
def fit_and_plot(target, features, model_name):
    # never let the target leak into the feature set
    features = [f for f in features if f != target]
    if not features:
        print('Select at least one feature that is not the target.')
        return

    X = df[features]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    if model_name == 'Linear Regression':
        model = LinearRegression()
    else:
        model = DecisionTreeRegressor(max_depth=3, random_state=42)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    lo = float(min(y_test.min(), y_pred.min()))
    hi = float(max(y_test.max(), y_pred.max()))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=y_test, y=y_pred, mode='markers',
        marker=dict(color='steelblue', size=9, opacity=0.8),
        name='Countries'))
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[lo, hi], mode='lines',
        line=dict(color='black', dash='dash'),
        name='Perfect prediction'))
    fig.update_layout(
        title=f'{model_name} — R²={r2:.3f} | MAE={mae:.2f} | RMSE={rmse:.2f}',
        xaxis_title=f'Actual {target}',
        yaxis_title=f'Predicted {target}',
        template='simple_white', width=650, height=550)
    fig.show()

### Launch the Explorer

`interactive_output` links the controls to the function without a "Run" button — every change re-fits the model and redraws the chart. Try dropping features, switching the target to `gdpp` or `income`, and flipping between the two models.

In [ ]:
controls = widgets.VBox([target_dropdown, feature_select, model_dropdown])
output = widgets.interactive_output(
    fit_and_plot,
    {'target': target_dropdown, 'features': feature_select, 'model_name': model_dropdown})

display(controls, output)

## Cross-Validation

A single train/test split gives just one estimate of performance — and that estimate can swing depending on *which* countries happened to land in the test set. **Cross-validation** rotates the held-out set across the whole dataset: it splits the data into several folds, trains on all but one, validates on the one left out, and repeats so every observation is used for validation exactly once. The result is a more stable, less luck-dependent picture of how the model generalizes.

### Visualizing How the Data Is Split

The clearest way to understand a splitter is to *see* it. For each fold we mark which samples are used for training versus validation. The function below builds that grid and renders it as a Plotly heatmap — one row per fold, one column per country.

In [ ]:
def plot_cv_indices(cv, n_samples):
    # 1 = validation sample, 0 = training sample, for each fold
    n_splits = cv.get_n_splits()
    matrix = np.zeros((n_splits, n_samples))
    for fold, (train_idx, test_idx) in enumerate(cv.split(np.arange(n_samples))):
        matrix[fold, test_idx] = 1

    fig = px.imshow(
        matrix, aspect='auto',
        color_continuous_scale=[[0, 'steelblue'], [1, 'darkorange']],
        labels=dict(x='Sample index', y='', color='Set'))
    fig.update_yaxes(tickmode='array',
                     tickvals=list(range(n_splits)),
                     ticktext=[f'Fold {i + 1}' for i in range(n_splits)])
    fig.update_layout(
        title='Cross-Validation Folds (orange = validation, blue = train)',
        coloraxis_showscale=False,
        template='simple_white', width=850, height=350)
    fig.show()

### Explore Different CV Strategies

Different splitters partition the data differently:

- **KFold** — contiguous, non-overlapping blocks; each fold is one block.
- **KFold (shuffled)** — the same, but rows are shuffled first, so folds aren't tied to the data's original order.
- **ShuffleSplit** — draws a fresh random validation subset each split, so (unlike KFold) validation sets can overlap across splits.

Switch strategies and slide the fold count to watch the pattern change.

In [ ]:
def show_cv(strategy, n_splits):
    if strategy == 'KFold':
        cv = KFold(n_splits=n_splits)
    elif strategy == 'KFold (shuffled)':
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:  # ShuffleSplit
        cv = ShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=42)
    plot_cv_indices(cv, len(df))


cv_strategy = widgets.Dropdown(
    options=['KFold', 'KFold (shuffled)', 'ShuffleSplit'],
    value='KFold', description='CV type:')
n_splits_slider = widgets.IntSlider(
    value=5, min=2, max=10, description='Splits:')

cv_controls = widgets.VBox([cv_strategy, n_splits_slider])
cv_output = widgets.interactive_output(
    show_cv, {'strategy': cv_strategy, 'n_splits': n_splits_slider})

display(cv_controls, cv_output)

### Comparing Models with Cross-Validation

Beyond visualizing the folds, cross-validation gives us a score *per fold*. `cross_val_score` runs the whole fit-and-evaluate loop for us. Plotting the spread of fold R² scores for each model shows not just which is better on average, but how consistent each one is — a wide box means performance depends heavily on which countries were held out.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

X_all = df.drop(columns='life_expec')
y_all = df['life_expec']

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=3, random_state=42),
}

fig = go.Figure()
for name, model in models.items():
    fold_scores = cross_val_score(model, X_all, y_all, cv=cv, scoring='r2')
    fig.add_trace(go.Box(y=fold_scores, name=name, boxpoints='all', jitter=0.4))
    print(f'{name:<20} mean R² = {fold_scores.mean():.3f} ± {fold_scores.std():.3f}')

fig.update_layout(
    title='5-Fold Cross-Validated R² by Model',
    yaxis_title='R² per fold',
    template='simple_white', width=650, height=450)
fig.show()